In [4]:
import pandas as pd

In [5]:
from google.colab import drive
drive.mount('/content/Drive')

Drive already mounted at /content/Drive; to attempt to forcibly remount, call drive.mount("/content/Drive", force_remount=True).


In [6]:
features = pd.read_csv('/content/Drive/MyDrive/Features data set.csv')
stores = pd.read_csv('/content/Drive/MyDrive/stores data-set.csv')
sales = pd.read_csv('/content/Drive/MyDrive/sales data-set.csv')

### Business Question
Which departments generate the highest and lowest total sales?

In [32]:
dept_sales = (
    sales.groupby('Dept')['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)

dept_sales.head()

,Weekly_Sales
Dept,
92,4.839433e+08
95,4.493202e+08
38,3.931181e+08
72,3.057252e+08
90,2.910685e+08


### Insight

**Insight:** Department 92 leads total sales at approximately $483.94M, followed by Departments 95 and 38.

### Business Question

Which departments generate the lowest total sales?

In [8]:
dept_sales = sales.groupby('Dept')['Weekly_Sales'].sum()
dept_sales.sort_values(ascending=True).head()

,Weekly_Sales
Dept,
47,-4962.93
43,14.32
39,177.98
78,1714.71
51,30572.83


### Insight

**Insight:** Department 47 recorded negative total sales of $4,962.93, while Departments 43 and 39 generated minimal sales.

### Business Question

Which departments have the highest average weekly sales?

In [33]:
dept_avg = sales.groupby('Dept')['Weekly_Sales'].mean()
dept_avg.sort_values(ascending = False).head().round(2)

,Weekly_Sales
Dept,
92,75204.87
95,69824.42
38,61090.62
72,50566.52
65,45441.71


### Insight

**Insight:** Department 92 leads average weekly sales at 75,204.87, followed by Departments 95 and 38.

### Business Question

Which departments have the lowest average weekly sales?

In [34]:
dept_avg = sales.groupby('Dept')['Weekly_Sales'].mean().reset_index(name='dept_avg')
dept_avg.sort_values(by='dept_avg',ascending = True).head().round(2)

,Dept,dept_avg
45,47,-7.68
41,43,1.19
63,78,7.30
37,39,11.12
49,51,21.93


### Insight

**Insight:** Department 47 has the lowest average weekly sales at -7.68, while Departments 43 and 78 also show minimal average sales.

### Business Question

Which departments have the highest variability in weekly sales?

In [35]:
dept_std = sales.groupby('Dept')['Weekly_Sales'].std()
dept_std.sort_values(ascending=False).head().round(2)

,Weekly_Sales
Dept,
92,49413.73
72,44710.98
95,38200.54
90,32462.02
7,27985.08


### Insight

**Insight:** Department 92 has the highest weekly sales variability at 49,413.73, followed by Departments 72 and 95, indicating greater week-to-week fluctuations.

### Business Question

Which departments have the lowest variability in weekly sales?

In [36]:
dept_std = sales.groupby('Dept')['Weekly_Sales'].std().reset_index(name='dept_std')
dept_std.sort_values(by='dept_std',ascending=True).head().set_index('Dept').round(2)

,dept_std
Dept,
43,1.03
39,9.51
78,10.40
45,24.96
51,29.15


### Insight

**Insight:** Department 43 has the lowest sales variability at 1.03, followed by Departments 39 and 78, indicating highly consistent weekly sales.

### Business Question

Which departments combine strong average sales with relatively low variability?

In [37]:
dept_risk = dept_avg.merge(dept_std,on='Dept')
strong_stable_dept = dept_risk.query(
    "dept_avg > 20000 & dept_std < 30000"
)
strong_stable_dept.sort_values(by='dept_avg',ascending=False).head().set_index('Dept').round(2)

,dept_avg,dept_std
Dept,,
38,61090.62,23966.91
65,45441.71,7086.89
40,44900.70,21903.88
2,43607.02,25176.76
91,33687.91,26247.46


### Insight

**Insight:** Department 38 leads the strong-and-stable group in average sales, while Department 65 combines strong sales with the lowest variability.

### Business Question

Which departments have the lowest relative variability in weekly sales?

In [38]:
dept_risk['cv'] = dept_risk['dept_std'] / dept_risk['dept_avg']
dept_risk.sort_values(by='cv', ascending=True).head().round(2)

,Dept,dept_avg,dept_std,cv
45,47,-7.68,307.88,-40.08
57,65,45441.71,7086.89,0.16
35,37,3111.08,887.65,0.29
36,38,61090.62,23966.91,0.39
38,40,44900.70,21903.88,0.49


In [39]:
dept_risk_positive = dept_risk.query("dept_avg > 0")

print(
    dept_risk_positive
    .sort_values(by='cv', ascending=False)
    .head().round(2)
)

    Dept  dept_avg  dept_std    cv
55    59    694.46   1991.36  2.87
80    99    415.49   1099.73  2.65
16    18   7336.94  13837.76  1.89
53    56   3833.71   5978.35  1.56
63    78      7.30     10.40  1.43


### Insight

**Insight:** Departments 59 and 99 have the highest relative variability, but their low average sales suggest that CV should be interpreted alongside sales volume.

### Business Question

Which departments have the highest number of negative sales records?

In [41]:
negative_dept = sales[sales['Weekly_Sales']<0].groupby('Dept').size().reset_index(name='negative_counts')
negative_dept.sort_values(by='negative_counts',ascending=False).head().set_index('Dept')

,negative_counts
Dept,
47,254
18,180
54,146
19,87
94,77


### Insight

**Insight:** Department 47 has the highest number of negative-sales records at 254, followed by Departments 18 and 54.

### Business Question

Which departments have the largest total negative-sales amount?

In [17]:
negative_sales_amount = sales[sales['Weekly_Sales']<0].groupby('Dept')['Weekly_Sales'].sum().reset_index().sort_values(by='Weekly_Sales').head().set_index('Dept')
negative_sales_amount

,Weekly_Sales
Dept,
47,-53188.06
6,-5841.64
54,-4829.81
80,-4231.48
72,-3047.58


### Insight

**Insight:** Department 47 has the largest total negative-sales amount at -$53,188.06, substantially exceeding the other departments shown.

### Business Question

How do department sales vary across years?

In [18]:
sales['Date'] = pd.to_datetime(
    sales['Date'],dayfirst=True
)
sales['Year'] = sales['Date'].dt.year

In [43]:
dept_year_sales = sales.groupby(['Year','Dept'])['Weekly_Sales'].sum().reset_index()
dept_year_sales.sort_values(by=['Year','Dept'],ascending=[True,True]).set_index('Year')

,Dept,Weekly_Sales
Year,,
2010,1,4.284353e+07
2010,2,9.405293e+07
2010,3,2.509114e+07
2010,4,5.557729e+07
2010,5,5.177156e+07
2010,6,1.237954e+07
2010,7,5.842404e+07
2010,8,6.536836e+07
2010,9,4.312974e+07


In [20]:
top_dept_year = dept_year_sales.sort_values(by='Weekly_Sales',ascending=False).head(10)
top_dept_year.set_index('Year')

,Dept,Weekly_Sales
Year,,
2011,92,1.764763e+08
2011,95,1.615699e+08
2010,92,1.558376e+08
2012,92,1.516294e+08
2010,95,1.498584e+08
2011,38,1.432613e+08
2012,95,1.378918e+08
2010,38,1.288553e+08
2012,38,1.210016e+08


### Insight

**Insight:** Department 92 in 2011 had the highest Year–Department sales among the top 10 combinations. Departments 92, 95, and 38 repeatedly appear among the strongest combinations across years.

### Business Question

Which Year–Department combinations have the lowest total weekly sales?

In [21]:
top_dept_year = dept_year_sales.sort_values(by='Weekly_Sales',ascending=True).head(10)
top_dept_year.set_index('Year')

,Dept,Weekly_Sales
Year,,
2011,47,-3742.29
2010,47,-2315.95
2010,43,1.75
2011,39,1.98
2012,43,5.50
2011,43,7.07
2010,77,7.97
2010,39,38.92
2012,39,137.08


### Insight

**Insight:** Department 47 recorded the lowest Year–Department sales in both 2011 and 2010, with negative totals. Departments 43 and 39 also repeatedly appear among the weakest-performing combinations.

## Business Question

Which store–department combinations generate the highest total weekly sales?

In [49]:
store_dept_sales = sales.groupby(['Store','Dept'])['Weekly_Sales'].sum().reset_index().sort_values(by='Weekly_Sales',ascending=False)
store_dept_sales.head(10).set_index('Store')

,Dept,Weekly_Sales
Store,,
14,92,26101497.71
2,92,23572153.03
20,92,23542625.04
13,92,23170876.20
4,92,22789210.43
20,95,21537795.62
4,95,21054815.74
27,92,20952094.22
14,95,20655911.35


### Insight

**Insight:** Store 14 and Department 92 form the highest-selling Store–Department combination at 26,101,497.71. Department 92 dominates the top-performing combinations, appearing in 7 of the top 10.

## Business Question

Which store–department combinations generate the lowest total weekly sales?

In [23]:
store_dept_sales = sales.groupby(['Store','Dept'])['Weekly_Sales'].sum().reset_index().sort_values(by='Weekly_Sales',ascending=True)
store_dept_sales.head(10).set_index('Store')

,Dept,Weekly_Sales
Store,,
35,47,-3567.18
1,47,-2562.00
10,47,-2329.00
42,72,-1209.93
45,47,-1061.04
16,47,-1039.08
12,47,-965.78
34,47,-947.03
6,47,-756.42


### Insight

**Insight:** Store 35 and Department 47 have the lowest Store–Department sales at -3,567.18. Department 47 appears in 9 of the bottom 10 combinations, reinforcing its unusually poor sales performance.

## Business Question

Which department is the top-performing department within each store?

In [24]:
store_dept_sales.loc[
    store_dept_sales.groupby('Store')['Weekly_Sales'].idxmax()
].set_index('Store')

,Dept,Weekly_Sales
Store,,
1,92,19370632.64
2,92,23572153.03
3,38,15529566.07
4,92,22789210.43
5,38,7893570.24
6,92,14160545.90
7,72,6447844.96
8,95,9002059.31
9,38,11184287.78


### Insight

**Insight:** Department 92 is the leading department across many stores, while Departments 38 and 72 also emerge as top-performing departments in multiple stores.

## Buisness Question
Which departments contribute the most sales within each year?

In [25]:
dept_sales = sales.groupby(['Dept','Year'])['Weekly_Sales'].sum().reset_index()
dept_sales_year = dept_sales.loc[dept_sales.groupby('Year')['Weekly_Sales'].idxmax()].sort_values(by='Year')
dept_sales_year.set_index('Dept')

,Year,Weekly_Sales
Dept,,
92,2010,1.558376e+08
92,2011,1.764763e+08
92,2012,1.516294e+08


### Insight
Department 92 remained the top-performing department across all three years, reaching its highest sales in 2011 before declining in 2012.

### Business Question

Which departments had the largest increase or decrease in sales from 2010 to 2012?

In [26]:
dept_year_sales = sales.groupby(['Dept','Year'])['Weekly_Sales'].sum().unstack()
dept_year_sales

Year,2010,2011,2012
Dept,,,
1,4.284353e+07,4.534931e+07,3.544593e+07
2,9.405293e+07,1.015053e+08,8.505299e+07
3,2.509114e+07,2.701822e+07,2.378310e+07
4,5.557729e+07,6.056611e+07,5.100335e+07
5,5.177156e+07,5.036340e+07,3.347240e+07
...,...,...,...
95,1.498584e+08,1.615699e+08,1.378918e+08
96,2.350948e+07,2.675970e+07,2.356474e+07
97,2.948267e+07,3.261287e+07,2.740097e+07


In [27]:
dept_sales_change = dept_year_sales.assign(
    sales_change = lambda x: x[2012] - x[2010]
)[['sales_change']].sort_values('sales_change',ascending=False)
pd.set_option('display.max_rows', None)

dept_sales_change

Year,sales_change
Dept,
87,3776524.49
48,417173.51
99,94721.96
96,55262.71
77,8546.89
47,3411.26
39,98.16
43,3.75
78,-858.15


### Insight
Department 87 recorded the largest sales increase from 2010 to 2012, while Department 72 experienced the largest decline. Overall, most departments recorded lower sales in 2012 compared with 2010.

### Business Question
Which departments show the largest fluctuations in annual sales?

In [28]:
dept_year_std = (
    dept_year_sales
    .std(axis=1)
    .sort_values(ascending=False)
    .reset_index(name='yearly_sales_std')
)

dept_year_std

,Dept,yearly_sales_std
0,72,1.883571e+07
1,92,1.329808e+07
2,7,1.211009e+07
3,95,1.183930e+07
4,38,1.128942e+07
5,5,1.018290e+07
6,90,9.639842e+06
7,2,8.238258e+06
8,40,7.061695e+06
9,91,6.232114e+06


### Insight
Department 72 shows the highest variation in annual sales, followed by Departments 92 and 7. Department 43 shows the lowest variation across the three years.

### Business Question

Which departments combine high average sales with relatively low variability?

In [50]:
strong_stable_dept = (
    dept_risk
    .query("dept_avg > 20000 and dept_std < 30000")
    .sort_values('dept_avg', ascending=False).round(2)
)

strong_stable_dept

,Dept,dept_avg,dept_std,cv
36,38,61090.62,23966.91,0.39
57,65,45441.71,7086.89,0.16
38,40,44900.70,21903.88,0.49
1,2,43607.02,25176.76,0.58
72,91,33687.91,26247.46,0.78
75,94,33405.88,25405.56,0.76
12,13,30663.80,16237.53,0.53
7,8,30191.26,18189.72,0.60
74,93,27008.06,22593.65,0.84
3,4,25974.63,13261.14,0.51


### Insight
Several high-sales departments also show relatively high variability. Department 7 has the highest relative variability in this group, while Department 91 has the lowest CV among the selected departments.

### Buisness Question

Which departments have high sales but also high relative variability?

In [52]:
high_sales_high_variability = (
    dept_risk
    .query("dept_avg > 20000 and cv > 0.75")
    .sort_values('cv', ascending=False).round(2)
)

high_sales_high_variability

,Dept,dept_avg,dept_std,cv
6,7,24161.24,27985.08,1.16
4,5,21365.58,19988.45,0.94
60,72,50566.52,44710.98,0.88
74,93,27008.06,22593.65,0.84
8,9,20206.68,16695.73,0.83
72,91,33687.91,26247.46,0.78
75,94,33405.88,25405.56,0.76


### Insight
Among high-sales departments, Department 7 has the highest relative variability (CV 1.158). Department 72 also shows high variability while maintaining strong average weekly sales, indicating that these departments have less consistent sales performance.

## Buisness Question

Which departments consistently perform well across all three years?

In [31]:
dept_year_positive = dept_year_sales.gt(0).all(axis = 1)
consistant_departments = dept_year_sales[dept_year_positive]
consistant_departments

Year,2010,2011,2012
Dept,,,
1,4.284353e+07,4.534931e+07,3.544593e+07
2,9.405293e+07,1.015053e+08,8.505299e+07
3,2.509114e+07,2.701822e+07,2.378310e+07
4,5.557729e+07,6.056611e+07,5.100335e+07
5,5.177156e+07,5.036340e+07,3.347240e+07
6,1.237954e+07,1.008493e+07,5.956193e+06
7,5.842404e+07,5.920394e+07,3.784958e+07
8,6.536836e+07,7.003633e+07,5.887608e+07
9,4.312974e+07,4.508026e+07,4.018326e+07


### Insight
There are 99 departments